# Phần Mạnh (1/2) — Đọc Dữ Liệu Gốc và Chia Tập Train/Test Chuẩn

**Mục tiêu của notebook:**
1. Đọc bộ dữ liệu thô viễn thông `WA_Fn-UseC_-Telco-Customer-Churn.csv` từ thư mục `data/raw/`.
2. Khám phá nhanh cấu trúc bảng, kiểu dữ liệu và kiểm tra các bất thường (đặc biệt là 11 giá trị khoảng trắng trong cột `TotalCharges`).
3. Chuyển đổi `TotalCharges` về kiểu số thực (`float64`) để nhận diện giá trị thiếu (`NaN`).
4. Loại bỏ trường định danh `customerID` (không mang giá trị dự báo, tránh bùng nổ số chiều và tránh rò rỉ thông tin).
5. Thực hiện phân chia tập dữ liệu thành **Train (80%)** và **Test (20%)** có phân tầng (`stratify=y`) với `random_state=42` cố định duy nhất một lần cho toàn nhóm.
6. Lưu trữ các file `X_train_raw.csv`, `X_test_raw.csv`, `y_train.csv`, `y_test.csv` vào thư mục `data/processed/` để phục vụ làm dữ liệu đầu vào cho Luồng A và Luồng B.

## 1. Thiết lập môi trường và cấu hình dự án
Ta xác định thư mục gốc của dự án và nạp các hằng số cấu hình chuẩn (`RANDOM_STATE = 42`, `TEST_SIZE = 0.20`, `TARGET_COLUMN = 'Churn'`) từ `src.config` cùng hàm chia dữ liệu từ `src.split_data`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "src" / "config.py").exists(), "Không tìm thấy thư mục dự án"
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_STATE, TEST_SIZE, TARGET_COLUMN
from src.split_data import chia_du_lieu

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

assert RAW_PATH.exists(), f"Không tìm thấy dataset: {RAW_PATH}"

print("Thư mục dự án:", PROJECT_ROOT)
print("random_state:", RANDOM_STATE)
print("test_size:", TEST_SIZE)
print("Cột mục tiêu:", TARGET_COLUMN)

Thư mục dự án: C:\Users\admin\Desktop\Subject\Nhap_Mon_Hoc_May\BTL\BTL_New_Version\BTL_Nhap_Mon_Hoc_May_NHOM4
random_state: 42
test_size: 0.2
Cột mục tiêu: Churn


## 2. Đọc và kiểm tra cấu trúc dữ liệu thô
Tải tập dữ liệu từ `data/raw/` và hiển thị 5 dòng đầu tiên cùng kích thước tổng quát.

In [2]:
df = pd.read_csv(RAW_PATH)

display(df.head())
print("Số dòng, số cột:", df.shape)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Số dòng, số cột: (7043, 21)


## 3. Khám phá các thuộc tính, kiểu dữ liệu và giá trị thiếu ban đầu
Kiểm tra danh sách cột, kiểu dữ liệu và phát hiện các ô thiếu.

In [3]:
print("Tên các cột:")
print(df.columns.tolist())

print("\nKiểu dữ liệu và số ô có giá trị:")
df.info()

Tên các cột:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Kiểu dữ liệu và số ô có giá trị:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecur

In [4]:
print("Số ô thiếu pandas nhận diện:")
display(df.isna().sum())

print("Số ô TotalCharges chỉ chứa khoảng trắng:")
print(df["TotalCharges"].str.strip().eq("").sum())

Số ô thiếu pandas nhận diện:


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Số ô TotalCharges chỉ chứa khoảng trắng:
11


## 4. Ép kiểu dữ liệu cột TotalCharges
Cột `TotalCharges` có 11 dòng chỉ chứa khoảng trắng `" "`, dẫn đến việc pandas nhận diện cột này là kiểu chuỗi (`object`). Ta dùng `pd.to_numeric(..., errors='coerce')` để ép sang dạng số thực, biến các ô khoảng trắng thành `NaN`.

*Lưu ý:* Đây là các khách hàng mới đăng ký (`tenure = 0`), chưa phát sinh cước lũy kế. Bước này chỉ ép kiểu dữ liệu thuần túy, không học bất kỳ tham số thống kê nào từ dữ liệu, do đó hoàn toàn không gây rò rỉ thông tin (data leakage).

In [5]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce",
)

print("Kiểu dữ liệu:", df["TotalCharges"].dtype)
print("Số giá trị thiếu:", df["TotalCharges"].isna().sum())

Kiểu dữ liệu: float64
Số giá trị thiếu: 11


## 5. Kiểm tra dòng trùng lặp và phân phối biến mục tiêu Churn
Kiểm tra tính duy nhất của dữ liệu và phân tích tỷ lệ khách hàng rời mạng (`Churn = Yes`) so với khách hàng ở lại (`Churn = No`). Tỷ lệ phân phối là khoảng 73.4% `No` và 26.6% `Yes` (dữ liệu mất cân bằng).

In [6]:
print("Số dòng trùng lặp:", df.duplicated().sum())
print("Số customerID trùng lặp:", df["customerID"].duplicated().sum())

print("Số khách hàng theo Churn:")
display(df[TARGET_COLUMN].value_counts())

print("Tỷ lệ theo Churn (%):")
display(df[TARGET_COLUMN].value_counts(normalize=True).mul(100).round(2))

Số dòng trùng lặp: 0
Số customerID trùng lặp: 0
Số khách hàng theo Churn:


Churn
No     5174
Yes    1869
Name: count, dtype: int64

Tỷ lệ theo Churn (%):


Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64

## 6. Tách đặc trưng dự đoán (X) và nhãn mục tiêu (y)
- Loại bỏ cột `customerID`: Đây là mã định danh ngẫu nhiên của từng khách hàng, không chứa thông tin hành vi hữu ích và sẽ gây bùng nổ số chiều nếu mã hóa One-Hot.
- Tách `Churn` làm biến mục tiêu $y$, còn lại 19 thuộc tính tạo thành ma trận đặc trưng $X$.

In [7]:
X = df.drop(columns=[TARGET_COLUMN, "customerID"])
y = df[TARGET_COLUMN]

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)
print("Còn Churn trong X?", TARGET_COLUMN in X.columns)
print("Còn customerID trong X?", "customerID" in X.columns)

Kích thước X: (7043, 19)
Kích thước y: (7043,)
Còn Churn trong X? False
Còn customerID trong X? False


## 7. Phân chia tập Train / Test (80% / 20%) có phân tầng (stratify=y)
**Nguyên tắc cốt lõi về phòng chống Data Leakage:**
- Cả nhóm Mạnh, Đức, Hải, Hùng **bắt buộc dùng chung một lần chia duy nhất này** để đảm bảo đánh giá công bằng trên cùng tập Test.
- Sử dụng `stratify=y` nhằm giữ nguyên tỷ lệ phân phối lớp Churn ($\approx 26.5\%$ `Yes` và $\approx 73.5\%$ `No`) ở cả tập Train và tập Test.
- Cố định `random_state=42` để kết quả có thể tái lập 100%.

In [8]:
X_train, X_test, y_train, y_test = chia_du_lieu(X, y)

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.index.intersection(X_test.index).empty

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("Nhãn Train:", y_train.value_counts().to_dict())
print("Nhãn Test:", y_test.value_counts().to_dict())

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)
Nhãn Train: {'No': 4139, 'Yes': 1495}
Nhãn Test: {'No': 1035, 'Yes': 374}


## 8. Lưu dữ liệu thô đã chia vào thư mục data/processed/
Lưu trữ các tập Train/Test độc lập dưới dạng CSV để bàn giao cho các bước tiếp theo:
- Mạnh dùng `X_train_raw.csv` để làm EDA và xây dựng tiền xử lý Luồng A.
- Đức dùng dữ liệu này để thiết kế Feature Engineering cho Luồng B.
- Sử dụng `index=False` để tránh sinh thêm cột chỉ số thừa khi ghi file.

In [9]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(PROCESSED_DIR / "X_train_raw.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test_raw.csv", index=False)

y_train.to_frame(name=TARGET_COLUMN).to_csv(
    PROCESSED_DIR / "y_train.csv",
    index=False,
)
y_test.to_frame(name=TARGET_COLUMN).to_csv(
    PROCESSED_DIR / "y_test.csv",
    index=False,
)

print("Đã lưu vào:", PROCESSED_DIR)

Đã lưu vào: C:\Users\admin\Desktop\Subject\Nhap_Mon_Hoc_May\BTL\BTL_New_Version\BTL_Nhap_Mon_Hoc_May_NHOM4\data\processed


## 9. Kiểm tra và xác nhận tính toàn vẹn của dữ liệu đã lưu
Đọc lại các file CSV vừa lưu để kiểm tra kích thước dòng/cột và đảm bảo không có cột lạ phát sinh.

In [10]:
expected = {
    "X_train_raw.csv": (5634, 19),
    "X_test_raw.csv": (1409, 19),
    "y_train.csv": (5634, 1),
    "y_test.csv": (1409, 1),
}

for filename, expected_shape in expected.items():
    saved = pd.read_csv(PROCESSED_DIR / filename)
    assert saved.shape == expected_shape, filename
    assert not any(
        column.startswith("Unnamed:")
        for column in saved.columns
    ), filename
    print(filename, saved.shape, "— đạt")

X_train_raw.csv (5634, 19) — đạt
X_test_raw.csv (1409, 19) — đạt
y_train.csv (5634, 1) — đạt
y_test.csv (1409, 1) — đạt
